# Scenario 2 — revised fallthrough generator and validator

Preserves reference feature/tool mappings and response catalogs. Never emits terminal NOT_EXECUTED. A failed remediation falls through if fewer than two distinct SHAP features have been evaluated; terminal REMEDIATION_FAILED remains allowed after that budget is exhausted.


1. No terminal `NOT_EXECUTED` after an unconfirmed first diagnostic.



2. Remediation failure must fall through `REMEDIATION_FAILED` remains an allowed terminal outcome.



Generate a revised script addressing the above both

In [1]:
from pathlib import Path
from collections import Counter
import json
import random

COUNT = 3000
SEED = 42
OUTPUT_DIR = Path.cwd()
DATASET_PATH = OUTPUT_DIR / "scenario2_revised_fallthrough_3000_training.jsonl"
MANIFEST_PATH = OUTPUT_DIR / "scenario2_revised_fallthrough_3000_manifest.jsonl"
REPORT_PATH = OUTPUT_DIR / "scenario2_revised_fallthrough_3000_validation.json"

## 1. Feature → diagnostic/remediation mapping

The exact mapping from the supplied reference; each feature is kept together.

In [2]:
FEATURES = {
    'EppValue': [
        ['EXCLUSIVE', 'diagnostic', 'CheckLowEnergySettings'],
        ['EXCLUSIVE', 'remediation', 'SetLowEnergySettings'],
    ],
    'PowerPlan': [
        ['EXCLUSIVE', 'diagnostic', 'CheckLowEnergySettings'],
        ['EXCLUSIVE', 'remediation', 'SetLowEnergySettings'],
    ],
    'ac_connect_disconnect_events_24h': [
        ['EXCLUSIVE', 'diagnostic', 'GetBatteryChargeStateChangesCount'],
        ['EXCLUSIVE', 'remediation', 'NotifyBatteryChargeStateChanges'],
    ],
    'ac_usage_ratio_12h': [
        ['EXCLUSIVE', 'diagnostic', 'GetBatteryChargeStateChangesCount'],
        ['EXCLUSIVE', 'remediation', 'NotifyBatteryChargeStateChanges'],
    ],
    'ac_usage_ratio_1D': [
        ['EXCLUSIVE', 'diagnostic', 'GetBatteryChargeStateChangesCount'],
        ['EXCLUSIVE', 'remediation', 'NotifyBatteryChargeStateChanges'],
    ],
    'ac_usage_ratio_7d': [
        ['EXCLUSIVE', 'diagnostic', 'CheckHeavyACAdapterUsageSettings'],
        ['EXCLUSIVE', 'remediation', 'UpdateBIOSBatteryProfile'],
    ],
    'ACMode': [
        ['EXCLUSIVE', 'diagnostic', 'CheckLowEnergySettings'],
        ['EXCLUSIVE', 'remediation', 'SetLowEnergySettings'],
    ],
    'Active': [
        ['EXCLUSIVE', 'diagnostic', 'CheckLowEnergySettings'],
        ['EXCLUSIVE', 'remediation', 'SetLowEnergySettings'],
    ],
    'adaptive_brightness / content_adaptive_brightness': [
        ['EXCLUSIVE', 'diagnostic', 'CheckLowEnergySettings'],
        ['EXCLUSIVE', 'remediation', 'SetLowEnergySettings'],
    ],
    'AppName': [
        ['EXCLUSIVE', 'diagnostic', 'CheckHeavyBatteryUsageByApps'],
        ['EXCLUSIVE', 'remediation', 'NotifyHeavyBatteryUsageByApps'],
    ],
    'AttachedAdapterWattage': [
        ['EXCLUSIVE', 'diagnostic', 'GetACAdapterChargeWattageStatus'],
        ['EXCLUSIVE', 'remediation', 'NotifyAdapterChargeWattageIsNotSufficient'],
    ],
    'BatteryUsage_by_Background_App_15mins': [
        ['EXCLUSIVE', 'diagnostic', 'CheckHeavyBatteryUsageByApps'],
        ['EXCLUSIVE', 'remediation', 'NotifyHeavyBatteryUsageByApps'],
    ],
    'BatteryUsage_by_Background_App_30mins': [
        ['EXCLUSIVE', 'diagnostic', 'CheckHeavyBatteryUsageByApps'],
        ['EXCLUSIVE', 'remediation', 'NotifyHeavyBatteryUsageByApps'],
    ],
    'BatteryUsage_by_Foreground_App_15mins': [
        ['EXCLUSIVE', 'diagnostic', 'CheckHeavyBatteryUsageByApps'],
        ['EXCLUSIVE', 'remediation', 'NotifyHeavyBatteryUsageByApps'],
    ],
    'BatteryUsage_by_Foreground_App_30mins': [
        ['EXCLUSIVE', 'diagnostic', 'CheckHeavyBatteryUsageByApps'],
        ['EXCLUSIVE', 'remediation', 'NotifyHeavyBatteryUsageByApps'],
    ],
    'bios_microcode_revision': [
        ['EXCLUSIVE', 'diagnostic', 'UpdateDrivers'],
        ['EXCLUSIVE', 'remediation', 'UpdateDrivers'],
    ],
    'BIOS_pass': [
        ['SHARED', 'diagnostic', 'CheckBatteryChargeCap'],
        ['SHARED', 'remediation', 'SetBatteryChargeCap'],
        ['SHARED', 'diagnostic', 'CheckHeavyACAdapterUsageSettings'],
        ['SHARED', 'remediation', 'UpdateBIOSBatteryProfile'],
    ],
    'bios_setting_primarily_ac': [
        ['EXCLUSIVE', 'diagnostic', 'CheckHeavyACAdapterUsageSettings'],
        ['EXCLUSIVE', 'remediation', 'UpdateBIOSBatteryProfile'],
    ],
    'BiosFullVersion': [
        ['EXCLUSIVE', 'diagnostic', 'UpdateDrivers'],
        ['EXCLUSIVE', 'remediation', 'UpdateDrivers'],
    ],
    'Brightness': [
        ['EXCLUSIVE', 'diagnostic', 'CheckLowEnergySettings'],
        ['EXCLUSIVE', 'remediation', 'SetLowEnergySettings'],
    ],
    'brightness_percent': [
        ['EXCLUSIVE', 'diagnostic', 'CheckLowEnergySettings'],
        ['EXCLUSIVE', 'remediation', 'SetLowEnergySettings'],
    ],
    'Capability': [
        ['SHARED', 'diagnostic', 'GetACAdapterChargeWattageStatus'],
        ['SHARED', 'remediation', 'NotifyAdapterChargeWattageIsNotSufficient'],
        ['SHARED', 'diagnostic', 'CheckBatteryChargeCap'],
        ['SHARED', 'remediation', 'SetBatteryChargeCap'],
    ],
    'Causes': [
        ['EXCLUSIVE', 'diagnostic', 'GetACAdapterChargeWattageStatus'],
        ['EXCLUSIVE', 'remediation', 'NotifyAdapterChargeWattageIsNotSufficient'],
    ],
    'Characteristics': [
        ['EXCLUSIVE', 'diagnostic', 'UpdateDrivers'],
        ['EXCLUSIVE', 'remediation', 'UpdateDrivers'],
    ],
    'ChargeMode': [
        ['EXCLUSIVE', 'diagnostic', 'CheckBatteryChargeCap'],
        ['EXCLUSIVE', 'remediation', 'SetBatteryChargeCap'],
    ],
    'ChargerStatusAcPowerPresent': [
        ['EXCLUSIVE', 'diagnostic', 'GetACAdapterChargeWattageStatus'],
        ['EXCLUSIVE', 'remediation', 'NotifyAdapterChargeWattageIsNotSufficient'],
    ],
    'ChargeStatus': [
        ['SHARED', 'diagnostic', 'GetACAdapterChargeWattageStatus'],
        ['SHARED', 'remediation', 'NotifyAdapterChargeWattageIsNotSufficient'],
        ['SHARED', 'diagnostic', 'CheckHeavyBatteryUsageByApps'],
        ['SHARED', 'remediation', 'NotifyHeavyBatteryUsageByApps'],
        ['SHARED', 'diagnostic', 'UpdateDrivers'],
        ['SHARED', 'remediation', 'UpdateDrivers'],
        ['SHARED', 'diagnostic', 'CheckLowEnergySettings'],
        ['SHARED', 'remediation', 'SetLowEnergySettings'],
    ],
    'ConfigManagerErrorCode': [
        ['EXCLUSIVE', 'diagnostic', 'UpdateDrivers'],
        ['EXCLUSIVE', 'remediation', 'UpdateDrivers'],
    ],
    'CpuEnergy': [
        ['EXCLUSIVE', 'diagnostic', 'CheckHeavyBatteryUsageByApps'],
        ['EXCLUSIVE', 'remediation', 'NotifyHeavyBatteryUsageByApps'],
    ],
    'current_adapter_wattage': [
        ['EXCLUSIVE', 'diagnostic', 'GetACAdapterChargeWattageStatus'],
        ['EXCLUSIVE', 'remediation', 'NotifyAdapterChargeWattageIsNotSufficient'],
    ],
    'CurrentChargePercentage': [
        ['SHARED', 'diagnostic', 'GetACAdapterChargeWattageStatus'],
        ['SHARED', 'remediation', 'NotifyAdapterChargeWattageIsNotSufficient'],
        ['SHARED', 'diagnostic', 'CheckBatteryChargeCap'],
        ['SHARED', 'remediation', 'SetBatteryChargeCap'],
        ['SHARED', 'diagnostic', 'CheckHeavyBatteryUsageByApps'],
        ['SHARED', 'remediation', 'NotifyHeavyBatteryUsageByApps'],
        ['SHARED', 'diagnostic', 'UpdateDrivers'],
        ['SHARED', 'remediation', 'UpdateDrivers'],
        ['SHARED', 'diagnostic', 'CheckLowEnergySettings'],
        ['SHARED', 'remediation', 'SetLowEnergySettings'],
    ],
    'dark_mode': [
        ['EXCLUSIVE', 'diagnostic', 'CheckLowEnergySettings'],
        ['EXCLUSIVE', 'remediation', 'SetLowEnergySettings'],
    ],
    'DataLineCount': [
        ['EXCLUSIVE', 'diagnostic', 'CheckHeavyBatteryUsageByApps'],
        ['EXCLUSIVE', 'remediation', 'NotifyHeavyBatteryUsageByApps'],
    ],
    'DCMode': [
        ['EXCLUSIVE', 'diagnostic', 'CheckLowEnergySettings'],
        ['EXCLUSIVE', 'remediation', 'SetLowEnergySettings'],
    ],
    'Description': [
        ['EXCLUSIVE', 'diagnostic', 'CheckBatteryChargeCap'],
        ['EXCLUSIVE', 'remediation', 'SetBatteryChargeCap'],
    ],
    'design_capacity_in_mwh': [
        ['EXCLUSIVE', 'diagnostic', 'CheckBatteryChargeCap'],
        ['EXCLUSIVE', 'remediation', 'SetBatteryChargeCap'],
    ],
    'DesignPowerDynamicInWatts': [
        ['EXCLUSIVE', 'diagnostic', 'GetACAdapterChargeWattageStatus'],
        ['EXCLUSIVE', 'remediation', 'NotifyAdapterChargeWattageIsNotSufficient'],
    ],
    'DesignPowerStaticInWatts': [
        ['EXCLUSIVE', 'diagnostic', 'GetACAdapterChargeWattageStatus'],
        ['EXCLUSIVE', 'remediation', 'NotifyAdapterChargeWattageIsNotSufficient'],
    ],
    'DeviceID': [
        ['EXCLUSIVE', 'diagnostic', 'UpdateDrivers'],
        ['EXCLUSIVE', 'remediation', 'UpdateDrivers'],
    ],
    'DischargeRateInmW': [
        ['SHARED', 'diagnostic', 'GetACAdapterChargeWattageStatus'],
        ['SHARED', 'remediation', 'NotifyAdapterChargeWattageIsNotSufficient'],
        ['SHARED', 'diagnostic', 'CheckBatteryChargeCap'],
        ['SHARED', 'remediation', 'SetBatteryChargeCap'],
        ['SHARED', 'diagnostic', 'CheckHeavyBatteryUsageByApps'],
        ['SHARED', 'remediation', 'NotifyHeavyBatteryUsageByApps'],
        ['SHARED', 'diagnostic', 'UpdateDrivers'],
        ['SHARED', 'remediation', 'UpdateDrivers'],
        ['SHARED', 'diagnostic', 'CheckLowEnergySettings'],
        ['SHARED', 'remediation', 'SetLowEnergySettings'],
    ],
    'DiskEnergy': [
        ['EXCLUSIVE', 'diagnostic', 'CheckHeavyBatteryUsageByApps'],
        ['EXCLUSIVE', 'remediation', 'NotifyHeavyBatteryUsageByApps'],
    ],
    'DisplayEnergy': [
        ['EXCLUSIVE', 'diagnostic', 'CheckHeavyBatteryUsageByApps'],
        ['EXCLUSIVE', 'remediation', 'NotifyHeavyBatteryUsageByApps'],
    ],
    'DisplaySettingsInfo': [
        ['EXCLUSIVE', 'diagnostic', 'CheckLowEnergySettings'],
        ['EXCLUSIVE', 'remediation', 'SetLowEnergySettings'],
    ],
    'DisplaySleepTime': [
        ['EXCLUSIVE', 'diagnostic', 'CheckLowEnergySettings'],
        ['EXCLUSIVE', 'remediation', 'SetLowEnergySettings'],
    ],
    'DriverVersion': [
        ['EXCLUSIVE', 'diagnostic', 'UpdateDrivers'],
        ['EXCLUSIVE', 'remediation', 'UpdateDrivers'],
    ],
    'DynamicEppEnabled': [
        ['EXCLUSIVE', 'diagnostic', 'CheckLowEnergySettings'],
        ['EXCLUSIVE', 'remediation', 'SetLowEnergySettings'],
    ],
    'EffectivePowerMode': [
        ['SHARED', 'diagnostic', 'CheckBatteryChargeCap'],
        ['SHARED', 'remediation', 'SetBatteryChargeCap'],
        ['SHARED', 'diagnostic', 'CheckLowEnergySettings'],
        ['SHARED', 'remediation', 'SetLowEnergySettings'],
    ],
    'EnergySettingsInfo': [
        ['EXCLUSIVE', 'diagnostic', 'CheckLowEnergySettings'],
        ['EXCLUSIVE', 'remediation', 'SetLowEnergySettings'],
    ],
    'ErrorCode': [
        ['SHARED', 'diagnostic', 'CheckBatteryChargeCap'],
        ['SHARED', 'remediation', 'SetBatteryChargeCap'],
        ['SHARED', 'diagnostic', 'CheckHeavyACAdapterUsageSettings'],
        ['SHARED', 'remediation', 'UpdateBIOSBatteryProfile'],
    ],
    'esf_diagnostic_version': [
        ['EXCLUSIVE', 'diagnostic', 'UpdateDrivers'],
        ['EXCLUSIVE', 'remediation', 'UpdateDrivers'],
    ],
    'expected_wattage': [
        ['EXCLUSIVE', 'diagnostic', 'GetACAdapterChargeWattageStatus'],
        ['EXCLUSIVE', 'remediation', 'NotifyAdapterChargeWattageIsNotSufficient'],
    ],
    'FastStartupEnabled': [
        ['EXCLUSIVE', 'diagnostic', 'CheckLowEnergySettings'],
        ['EXCLUSIVE', 'remediation', 'SetLowEnergySettings'],
    ],
    'FileSizeInKB': [
        ['EXCLUSIVE', 'diagnostic', 'CheckHeavyBatteryUsageByApps'],
        ['EXCLUSIVE', 'remediation', 'NotifyHeavyBatteryUsageByApps'],
    ],
    'FirewallSettingsInfo': [
        ['EXCLUSIVE', 'diagnostic', 'CheckLowEnergySettings'],
        ['EXCLUSIVE', 'remediation', 'SetLowEnergySettings'],
    ],
    'FirstGotFocusTimestamp': [
        ['EXCLUSIVE', 'diagnostic', 'CheckHeavyBatteryUsageByApps'],
        ['EXCLUSIVE', 'remediation', 'NotifyHeavyBatteryUsageByApps'],
    ],
    'full_charge_capacity_in_mwh': [
        ['EXCLUSIVE', 'diagnostic', 'CheckBatteryChargeCap'],
        ['EXCLUSIVE', 'remediation', 'SetBatteryChargeCap'],
    ],
    'FullExeName': [
        ['EXCLUSIVE', 'diagnostic', 'CheckHeavyBatteryUsageByApps'],
        ['EXCLUSIVE', 'remediation', 'NotifyHeavyBatteryUsageByApps'],
    ],
    'GraphicsSettingsInfo': [
        ['EXCLUSIVE', 'diagnostic', 'CheckLowEnergySettings'],
        ['EXCLUSIVE', 'remediation', 'SetLowEnergySettings'],
    ],
    'hdr_enabled': [
        ['EXCLUSIVE', 'diagnostic', 'CheckLowEnergySettings'],
        ['EXCLUSIVE', 'remediation', 'SetLowEnergySettings'],
    ],
    'Impacts': [
        ['EXCLUSIVE', 'diagnostic', 'GetACAdapterChargeWattageStatus'],
        ['EXCLUSIVE', 'remediation', 'NotifyAdapterChargeWattageIsNotSufficient'],
    ],
    'InstallDate': [
        ['EXCLUSIVE', 'diagnostic', 'UpdateDrivers'],
        ['EXCLUSIVE', 'remediation', 'UpdateDrivers'],
    ],
    'IOReadBytesPerSec': [
        ['EXCLUSIVE', 'diagnostic', 'CheckHeavyBatteryUsageByApps'],
        ['EXCLUSIVE', 'remediation', 'NotifyHeavyBatteryUsageByApps'],
    ],
    'IOWriteBytesPerSec': [
        ['EXCLUSIVE', 'diagnostic', 'CheckHeavyBatteryUsageByApps'],
        ['EXCLUSIVE', 'remediation', 'NotifyHeavyBatteryUsageByApps'],
    ],
    'is_portable_battery_active': [
        ['EXCLUSIVE', 'diagnostic', 'CheckBatteryChargeCap'],
        ['EXCLUSIVE', 'remediation', 'SetBatteryChargeCap'],
    ],
    'IsAcPresent': [
        ['EXCLUSIVE', 'diagnostic', 'GetACAdapterChargeWattageStatus'],
        ['EXCLUSIVE', 'remediation', 'NotifyAdapterChargeWattageIsNotSufficient'],
    ],
    'IsPsidValid': [
        ['EXCLUSIVE', 'diagnostic', 'GetACAdapterChargeWattageStatus'],
        ['EXCLUSIVE', 'remediation', 'NotifyAdapterChargeWattageIsNotSufficient'],
    ],
    'ItemRequirement': [
        ['EXCLUSIVE', 'diagnostic', 'UpdateDrivers'],
        ['EXCLUSIVE', 'remediation', 'UpdateDrivers'],
    ],
    'ItemStatus': [
        ['EXCLUSIVE', 'diagnostic', 'UpdateDrivers'],
        ['EXCLUSIVE', 'remediation', 'UpdateDrivers'],
    ],
    'keyboard_backlight': [
        ['EXCLUSIVE', 'diagnostic', 'CheckLowEnergySettings'],
        ['EXCLUSIVE', 'remediation', 'SetLowEnergySettings'],
    ],
    'KeyboardSettingsInfo': [
        ['EXCLUSIVE', 'diagnostic', 'CheckLowEnergySettings'],
        ['EXCLUSIVE', 'remediation', 'SetLowEnergySettings'],
    ],
    'LastDefragmentTime': [
        ['EXCLUSIVE', 'diagnostic', 'CheckLowEnergySettings'],
        ['EXCLUSIVE', 'remediation', 'SetLowEnergySettings'],
    ],
    'LastLostFocusTimestamp': [
        ['EXCLUSIVE', 'diagnostic', 'CheckHeavyBatteryUsageByApps'],
        ['EXCLUSIVE', 'remediation', 'NotifyHeavyBatteryUsageByApps'],
    ],
    'LightModeSettingsInfo': [
        ['EXCLUSIVE', 'diagnostic', 'CheckLowEnergySettings'],
        ['EXCLUSIVE', 'remediation', 'SetLowEnergySettings'],
    ],
    'MbbEnergy': [
        ['EXCLUSIVE', 'diagnostic', 'CheckHeavyBatteryUsageByApps'],
        ['EXCLUSIVE', 'remediation', 'NotifyHeavyBatteryUsageByApps'],
    ],
    'md_os_version': [
        ['EXCLUSIVE', 'diagnostic', 'UpdateDrivers'],
        ['EXCLUSIVE', 'remediation', 'UpdateDrivers'],
    ],
    'MinimumRequiredAdapterWattage': [
        ['EXCLUSIVE', 'diagnostic', 'GetACAdapterChargeWattageStatus'],
        ['EXCLUSIVE', 'remediation', 'NotifyAdapterChargeWattageIsNotSufficient'],
    ],
    'Name': [
        ['SHARED', 'diagnostic', 'GetACAdapterChargeWattageStatus'],
        ['SHARED', 'remediation', 'NotifyAdapterChargeWattageIsNotSufficient'],
        ['SHARED', 'diagnostic', 'UpdateDrivers'],
        ['SHARED', 'remediation', 'UpdateDrivers'],
        ['SHARED', 'diagnostic', 'CheckLowEnergySettings'],
        ['SHARED', 'remediation', 'SetLowEnergySettings'],
    ],
    'NetworkEnergy': [
        ['EXCLUSIVE', 'diagnostic', 'CheckHeavyBatteryUsageByApps'],
        ['EXCLUSIVE', 'remediation', 'NotifyHeavyBatteryUsageByApps'],
    ],
    'NetworkSentTotalInBytes': [
        ['EXCLUSIVE', 'diagnostic', 'CheckHeavyBatteryUsageByApps'],
        ['EXCLUSIVE', 'remediation', 'NotifyHeavyBatteryUsageByApps'],
    ],
    'PeakShift': [
        ['EXCLUSIVE', 'diagnostic', 'CheckBatteryChargeCap'],
        ['EXCLUSIVE', 'remediation', 'SetBatteryChargeCap'],
    ],
    'power_mode': [
        ['EXCLUSIVE', 'diagnostic', 'CheckLowEnergySettings'],
        ['EXCLUSIVE', 'remediation', 'SetLowEnergySettings'],
    ],
    'PowerStatus': [
        ['EXCLUSIVE', 'diagnostic', 'CheckLowEnergySettings'],
        ['EXCLUSIVE', 'remediation', 'SetLowEnergySettings'],
    ],
    'ProcessId': [
        ['EXCLUSIVE', 'diagnostic', 'CheckHeavyBatteryUsageByApps'],
        ['EXCLUSIVE', 'remediation', 'NotifyHeavyBatteryUsageByApps'],
    ],
    'ProcessorTimeInSeconds': [
        ['EXCLUSIVE', 'diagnostic', 'CheckHeavyBatteryUsageByApps'],
        ['EXCLUSIVE', 'remediation', 'NotifyHeavyBatteryUsageByApps'],
    ],
    'ProcessorTimePercentage': [
        ['EXCLUSIVE', 'diagnostic', 'CheckHeavyBatteryUsageByApps'],
        ['EXCLUSIVE', 'remediation', 'NotifyHeavyBatteryUsageByApps'],
    ],
    'ProductName': [
        ['EXCLUSIVE', 'diagnostic', 'CheckHeavyBatteryUsageByApps'],
        ['EXCLUSIVE', 'remediation', 'NotifyHeavyBatteryUsageByApps'],
    ],
    'Publisher': [
        ['EXCLUSIVE', 'diagnostic', 'UpdateDrivers'],
        ['EXCLUSIVE', 'remediation', 'UpdateDrivers'],
    ],
    'ReleaseDate': [
        ['EXCLUSIVE', 'diagnostic', 'UpdateDrivers'],
        ['EXCLUSIVE', 'remediation', 'UpdateDrivers'],
    ],
    'screen_off_after_minutes': [
        ['EXCLUSIVE', 'diagnostic', 'CheckLowEnergySettings'],
        ['EXCLUSIVE', 'remediation', 'SetLowEnergySettings'],
    ],
    'ScreenSaverEnabled': [
        ['EXCLUSIVE', 'diagnostic', 'CheckLowEnergySettings'],
        ['EXCLUSIVE', 'remediation', 'SetLowEnergySettings'],
    ],
    'SocEnergy': [
        ['EXCLUSIVE', 'diagnostic', 'CheckHeavyBatteryUsageByApps'],
        ['EXCLUSIVE', 'remediation', 'NotifyHeavyBatteryUsageByApps'],
    ],
    'State': [
        ['SHARED', 'diagnostic', 'UpdateDrivers'],
        ['SHARED', 'remediation', 'UpdateDrivers'],
        ['SHARED', 'diagnostic', 'GetBatteryChargeStateChangesCount'],
        ['SHARED', 'remediation', 'NotifyBatteryChargeStateChanges'],
    ],
    'StateChange': [
        ['EXCLUSIVE', 'diagnostic', 'CheckHeavyACAdapterUsageSettings'],
        ['EXCLUSIVE', 'remediation', 'UpdateBIOSBatteryProfile'],
    ],
    'stop_usb_when_screen_off': [
        ['EXCLUSIVE', 'diagnostic', 'CheckLowEnergySettings'],
        ['EXCLUSIVE', 'remediation', 'SetLowEnergySettings'],
    ],
    'SystemSleepTime': [
        ['EXCLUSIVE', 'diagnostic', 'CheckLowEnergySettings'],
        ['EXCLUSIVE', 'remediation', 'SetLowEnergySettings'],
    ],
    'TestPass': [
        ['SHARED', 'diagnostic', 'CheckBatteryChargeCap'],
        ['SHARED', 'remediation', 'SetBatteryChargeCap'],
        ['SHARED', 'diagnostic', 'CheckHeavyACAdapterUsageSettings'],
        ['SHARED', 'remediation', 'UpdateBIOSBatteryProfile'],
    ],
    'top_consumers': [
        ['EXCLUSIVE', 'diagnostic', 'CheckHeavyBatteryUsageByApps'],
        ['EXCLUSIVE', 'remediation', 'NotifyHeavyBatteryUsageByApps'],
    ],
    'TotalOnFocusDurationInSeconds': [
        ['EXCLUSIVE', 'diagnostic', 'CheckHeavyBatteryUsageByApps'],
        ['EXCLUSIVE', 'remediation', 'NotifyHeavyBatteryUsageByApps'],
    ],
    'ValidationCode': [
        ['EXCLUSIVE', 'diagnostic', 'CheckBatteryChargeCap'],
        ['EXCLUSIVE', 'remediation', 'SetBatteryChargeCap'],
    ],
    'Vendor': [
        ['EXCLUSIVE', 'diagnostic', 'UpdateDrivers'],
        ['EXCLUSIVE', 'remediation', 'UpdateDrivers'],
    ],
    'Version': [
        ['EXCLUSIVE', 'diagnostic', 'UpdateDrivers'],
        ['EXCLUSIVE', 'remediation', 'UpdateDrivers'],
    ],
    'VideoDimmingSupported': [
        ['EXCLUSIVE', 'diagnostic', 'CheckLowEnergySettings'],
        ['EXCLUSIVE', 'remediation', 'SetLowEnergySettings'],
    ],
    'WirelessSettingsInfo': [
        ['EXCLUSIVE', 'diagnostic', 'CheckLowEnergySettings'],
        ['EXCLUSIVE', 'remediation', 'SetLowEnergySettings'],
    ],
    'WorkingSetInBytes': [
        ['EXCLUSIVE', 'diagnostic', 'CheckHeavyBatteryUsageByApps'],
        ['EXCLUSIVE', 'remediation', 'NotifyHeavyBatteryUsageByApps'],
    ],
}

## 2. Diagnostic → remediation mapping

In [3]:
PAIR = {
    "GetBatteryChargeStateChangesCount": "NotifyBatteryChargeStateChanges",
    "CheckHeavyBatteryUsageByApps": "NotifyHeavyBatteryUsageByApps",
    "CheckLowEnergySettings": "SetLowEnergySettings",
    "CheckBatteryChargeCap": "SetBatteryChargeCap",
    "CheckHeavyACAdapterUsageSettings": "UpdateBIOSBatteryProfile",
    "GetACAdapterChargeWattageStatus": "NotifyAdapterChargeWattageIsNotSufficient",
    "UpdateDrivers": "UpdateDrivers"
}

## 3. Diagnostic responses

Exact strings and labels from the supplied reference.

In [4]:
DIAG = {
    'GetACAdapterChargeWattageStatus': [
        {"status": 'NEGATIVE', "message": 'Sufficient AC Adapter charge wattage detected.No need to display AC Adapter Charge Wattage Notification.'},
        {"status": 'POSITIVE', "message": 'Insufficient AC Adapter charge wattage detected. AC Adapter Charge Wattage Notification has to be displayed'},
        {"status": 'NEGATIVE', "message": 'Adapter wattage data not found in DTP. No need to display AC Adapter Charge Wattage Notification.'},
        {"status": 'ERROR', "message": 'Some error occurred while executing GetACAdapterChargeWattageStatus. No need to display AC Adapter Charge Wattage Notification.'},
        {"status": 'COOLDOWN', "message": 'Tool is currently disabled or on cooldown'},
    ],
    'CheckBatteryChargeCap': [
        {"status": 'POSITIVE', "message": 'Battery charge cap is enabled.'},
        {"status": 'NEGATIVE', "message": 'Battery charge cap is disabled.'},
        {"status": 'ERROR', "message": 'Error checking battery charge cap status.'},
        {"status": 'COOLDOWN', "message": 'Tool is currently disabled or on cooldown'},
    ],
    'CheckLowEnergySettings': [
        {"status": 'NEGATIVE', "message": 'Device is not on battery.'},
        {"status": 'NEGATIVE', "message": 'Low energy settings are enabled.'},
        {"status": 'POSITIVE', "message": 'Low energy settings are disabled.'},
        {"status": 'ERROR', "message": 'Unable to detect low energy settings. Script execution failed.'},
        {"status": 'COOLDOWN', "message": 'Tool is currently disabled or on cooldown'},
    ],
    'GetBatteryChargeStateChangesCount': [
        {"status": 'POSITIVE', "message": 'Frequent AC Power Connect/Disconnect.'},
        {"status": 'NEGATIVE', "message": 'No frequent AC Power Connect/Disconnect detected.'},
        {"status": 'POSITIVE', "message": 'High rate of charge state changes detected. Battery charge state notification has to be displayed'},
        {"status": 'NEGATIVE', "message": 'No high rate of charge state changes detected.No need to display Battery charge state notification.'},
        {"status": 'ERROR', "message": 'Some error occurred while executing GetBatteryChargeStateChangesCount. No need to display Battery charge state notification.'},
        {"status": 'COOLDOWN', "message": 'Tool is currently disabled or on cooldown'},
    ],
    'CheckHeavyACAdapterUsageSettings': [
        {"status": 'POSITIVE', "message": 'Device is primarily used on AC adapter. Remediation is required.'},
        {"status": 'NEGATIVE', "message": 'Device is not primarily used on AC adapter.No need to display Battery charge state notification.'},
        {"status": 'ERROR', "message": 'Some error occurred while collecting AC adapter usage data. No need to display Battery charge state notification.'},
        {"status": 'COOLDOWN', "message": 'Tool is currently disabled or on cooldown'},
    ],
    'CheckHeavyBatteryUsageByApps': [
        {"status": 'POSITIVE', "message": 'High battery usage was detected.'},
        {"status": 'NEGATIVE', "message": 'Battery usage is below the threshold.'},
        {"status": 'ERROR', "message": 'Some error occurred while executing the script.'},
        {"status": 'ERROR', "message": 'Some error occurred while checking heavy battery usage.'},
        {"status": 'COOLDOWN', "message": 'Tool is currently disabled or on cooldown'},
    ],
    'UpdateDrivers': [
        {"status": 'POSITIVE', "message": 'Driver update has completed.'},
        {"status": 'POSITIVE', "message": 'Driver update has completed. Consent to restart the machine has been received.'},
        {"status": 'POSITIVE', "message": 'Notification displayed but user did not approve for remediation - Displayed'},
        {"status": 'POSITIVE', "message": 'Notification displayed but user did not approve for remediation - DisplayedWithoutAction'},
        {"status": 'POSITIVE', "message": 'Notification displayed but user did not approve for remediation - SecondaryAction'},
        {"status": 'POSITIVE', "message": 'Notification displayed but user did not approve for remediation - UserCanceled'},
        {"status": 'POSITIVE', "message": 'Notification displayed but user did not approve for remediation - Terminated'},
        {"status": 'POSITIVE', "message": 'Driver update has completed. But consent to restart the machine has not been received. Notification timed out.'},
        {"status": 'POSITIVE', "message": 'Notification sent directly to the Windows Notification Center without a pop up'},
        {"status": 'POSITIVE', "message": 'Remediation complete. Could not request reboot consent because Windows notifications are disabled.'},
        {"status": 'POSITIVE', "message": 'Remediation complete. Could not request reboot consent because application was hidden.'},
        {"status": 'POSITIVE', "message": 'Driver update has completed. But consent to restart the machine has not been received. No user logged in.'},
        {"status": 'POSITIVE', "message": 'Driver update has completed. But consent to restart the machine has not been received. System locked.'},
        {"status": 'POSITIVE', "message": 'Driver update has completed. But consent to restart the machine has not been received. Notification failed.'},
        {"status": 'POSITIVE', "message": 'Driver update has completed. But consent to restart the machine has not been received.'},
        {"status": 'NEGATIVE', "message": 'No drivers found for installation after driver scan.'},
        {"status": 'ERROR', "message": 'Failed to install drivers.'},
        {"status": 'ERROR', "message": 'Failed to scan drivers.'},
        {"status": 'NEGATIVE', "message": 'Could not perform driver scan as AutoDriverUpdate or CustomCatalog is enabled in SupportAssist'},
        {"status": 'NEGATIVE', "message": 'Driver scan could not be performed as device is not ready for scan. State: {0}'},
        {"status": 'NEGATIVE', "message": 'Driver install could not be performed as device is not ready for scan. State: {0}'},
        {"status": 'NEGATIVE', "message": 'Driver scan could not be performed as Bradbury plugin failed to load'},
        {"status": 'NEGATIVE', "message": 'Driver install could not be performed as Bradbury plugin failed to load'},
        {"status": 'NEGATIVE', "message": 'No drivers have been updated.'},
        {"status": 'COOLDOWN', "message": 'Tool is currently disabled or on cooldown'},
    ],
}

## 4. Remediation responses

Exact strings and labels from the supplied reference.

In [5]:
REM = {
    'NotifyAdapterChargeWattageIsNotSufficient': [
        {"status": 'SUCCESS', "message": 'AC Adapter Charge Wattage Notification displayed successfully to user.'},
        {"status": 'SUCCESS', "message": 'Notification sent directly to the Windows Notification Center without a pop up'},
        {"status": 'NEGATIVE', "message": 'Notification could not be displayed because Windows notifications are disabled in system'},
        {"status": 'NEGATIVE', "message": 'Notification could not be displayed because no user is logged in'},
        {"status": 'NEGATIVE', "message": 'Notification could not be displayed because user system is locked'},
        {"status": 'ERROR', "message": 'Failed to display AC Adapter Charge Wattage Notification.'},
        {"status": 'NEGATIVE', "message": 'Adapter wattage data not found in DTP.'},
        {"status": 'COOLDOWN', "message": 'Tool is currently disabled or on cooldown'},
        {"status": 'ERROR', "message": 'Some error occurred while executing NotifyAdapterChargeWattageIsNotSufficient.'},
    ],
    'SetBatteryChargeCap': [
        {"status": 'SUCCESS', "message": 'BIOS battery charge level settings were updated. Remediation was applied.'},
        {"status": 'ERROR', "message": "Couldn't update BIOS battery charge level settings due to a script error."},
        {"status": 'ERROR', "message": 'Some error occurred while executing the tool RemediateBatteryChargeCap.'},
        {"status": 'COOLDOWN', "message": 'Tool is currently disabled or on cooldown'},
    ],
    'SetLowEnergySettings': [
        {"status": 'SUCCESS', "message": 'All settings for low energy are optimized.'},
        {"status": 'NEGATIVE', "message": 'Notification displayed but user did not approve for remediation - Displayed'},
        {"status": 'NEGATIVE', "message": 'Notification displayed but user did not approve for remediation - DisplayedWithoutAction'},
        {"status": 'NEGATIVE', "message": 'Notification displayed but user did not approve for remediation - SecondaryAction'},
        {"status": 'NEGATIVE', "message": 'Notification displayed but user did not approve for remediation - UserCanceled'},
        {"status": 'NEGATIVE', "message": 'Notification displayed but user did not approve for remediation - Terminated'},
        {"status": 'NEGATIVE', "message": 'Notification displayed but user did not approve for remediation - TimedOut'},
        {"status": 'NEGATIVE', "message": 'Notification displayed but user did not approve for remediation - Suppressed'},
        {"status": 'NEGATIVE', "message": 'Notification could not be displayed because Windows notifications are disabled in system'},
        {"status": 'NEGATIVE', "message": 'Notification could not be displayed because no user is logged in'},
        {"status": 'NEGATIVE', "message": 'Notification could not be displayed because user system is locked'},
        {"status": 'ERROR', "message": 'Could not display low energy settings notification.'},
        {"status": 'ERROR', "message": 'Unable to set low energy settings.'},
        {"status": 'COOLDOWN', "message": 'Tool is currently disabled or on cooldown'},
    ],
    'NotifyBatteryChargeStateChanges': [
        {"status": 'SUCCESS', "message": 'Battery Charge State Notification displayed successfully to user.'},
        {"status": 'SUCCESS', "message": 'Notification sent directly to the Windows Notification Center without a pop up'},
        {"status": 'NEGATIVE', "message": 'Notification could not be displayed because Windows notifications are disabled in system'},
        {"status": 'NEGATIVE', "message": 'Notification could not be displayed because no user is logged in'},
        {"status": 'NEGATIVE', "message": 'Notification could not be displayed because user system is locked'},
        {"status": 'ERROR', "message": 'Failed to display Battery Charge State Notification.'},
        {"status": 'COOLDOWN', "message": 'Tool is currently disabled or on cooldown'},
    ],
    'UpdateBIOSBatteryProfile': [
        {"status": 'SUCCESS', "message": 'BIOS battery profile updated to primarily on AC power. Remediation successful.'},
        {"status": 'ERROR', "message": "Couldn't complete the BIOS battery profile update due to a script error."},
        {"status": 'ERROR', "message": 'Some error occurred while executing UpdateBIOSBatteryProfile.'},
        {"status": 'COOLDOWN', "message": 'Tool is currently disabled or on cooldown'},
    ],
    'NotifyHeavyBatteryUsageByApps': [
        {"status": 'SUCCESS', "message": 'User notified of top three apps consuming battery.'},
        {"status": 'SUCCESS', "message": 'Notification sent directly to the Windows Notification Center without a pop up'},
        {"status": 'NEGATIVE', "message": 'Notification could not be displayed because Windows notifications are disabled in system'},
        {"status": 'NEGATIVE', "message": 'Notification could not be displayed because no user is logged in'},
        {"status": 'NEGATIVE', "message": 'Notification could not be displayed because user system is locked'},
        {"status": 'ERROR', "message": 'Failed to notify user of top three apps consuming battery.'},
        {"status": 'NEGATIVE', "message": 'Battery usage is below the threshold.'},
        {"status": 'ERROR', "message": 'Some error occurred while executing the script.'},
        {"status": 'ERROR', "message": 'Some error occurred while showing user notification.'},
        {"status": 'COOLDOWN', "message": 'Tool is currently disabled or on cooldown'},
    ],
}

## 5. Generator helpers

Copied from the attached generator; no mapping or response changes.

In [6]:
UNSUPPORTED_LEGACY_PATTERNS = [
    'remediation_cooldown_blocked',
    'approach1_diagnostic_cooldown',
    'approach2_remediation_cooldown',
    'all_features_exhausted_with_cooldown',
]


### 5a. Mapping and response selectors

In [7]:
def mapped(feature):
    entries = FEATURES[feature]
    return [v for _, kind, v in entries if kind == 'diagnostic']

def feature_for(tool, classification=None, exclude=()):
    candidates = [f for f in FEATURES if tool in mapped(f) and f not in exclude and (classification is None or FEATURES[f][0][0] == classification)]
    if not candidates:
        raise ValueError(f'No feature for {tool} ({classification})')
    return sorted(candidates)[0]

def response(tool, status, rng, exact=None):
    choices = [x for x in DIAG[tool] if x['status'] == status]
    if exact is not None:
        choices = [x for x in choices if x['message'] == exact]
    if not choices:
        raise ValueError(f'No workbook response for {tool}/{status}')
    return rng.choice(choices)

def remediation_response(tool, status, rng, exact=None):
    choices = [x for x in REM[tool] if x['status'] == status]
    if exact is not None:
        choices = [x for x in choices if x['message'] == exact]
    if not choices:
        raise ValueError(f'No workbook remediation response for {tool}/{status}')
    return rng.choice(choices)

def choose_distinct(count, rng, statuses=None, exclude=()):
    tools = [t for t in DIAG if t != 'UpdateDrivers' and t not in exclude]
    if statuses:
        tools = [t for t in tools if all((any((r['status'] == st for r in DIAG[t])) for st in statuses))]
    rng.shuffle(tools)
    return tools[:count]

### 5b. ReAct message builder

In [8]:
# def build(pattern, steps, rng):
#     """steps: list of (feature, tool, response-dict, kind)."""
#     used = set()
#     for f, t, result, kind in steps:
#         assert f in FEATURES
#         assert t not in used, f'Repeated tool: {t}'
#         used.add(t)
#         if kind == 'diagnostic':
#             assert t in mapped(f)
#             assert result in DIAG[t]
#         else:
#             assert t in REM and result in REM[t]
#             assert any((k == 'remediation' and v == t for _, k, v in FEATURES[f]))
#     features = []
#     for f, _, _, _ in steps:
#         if f not in features:
#             features.append(f)
#     scores = [round(0.95 - i * 0.11, 3) for i in range(len(features))]
#     messages = [{'role': 'system', 'content': 'You are the Reasoning & Execution engine for Battery Quick Discharging issues. Your role is to analyze SHAP feature attributions, execute diagnostics & remediations when issues are confirmed, and report the final outcome. CORE RULES - REACT SEQUENCE: Number each step sequentially (THOUGHT N:, ACTION N:). Do not emit a subsequent THOUGHT until you receive the output of your ACTION. - TOOL CADENCE: Call only one tool per turn. Never repeat a diagnostic tool that has already run. - TOOL OBSERVATION: Observe the diagnostic tool output thoroughly and classify it as CONFIRMED or NOT CONFIRMED. If NOT CONFIRMED, fall through to the next SHAP-ranked feature. - AUTO-REMEDIATION: Once a diagnostic tool confirms a symptom, proceed immediately to invoke the corresponding remediation tool. REASONING PROCESS 1. Hypothesis & Diagnostics: Select the appropriate read-only diagnostic tool to verify the root cause, and execute it. 2. Evidence Evaluation & Action: Read the diagnostic observation. If an issue is verified, explain the underlying root cause and execute the corresponding corrective remediation tool (e.g., configuration fix, update, or user notification). If a diagnostic tool returns an error, fails to execute, or states the issue is unconfirmed, do not invoke a remediation tool. Enter a cool down state (do not retry the failed tool), log the failure, and proceed with the next feature based on SHAP ranking. 3. Resolution: Confirm the remediation outcome (or report the diagnostic failure) and output your final assessment matching the layout below. TOOL FORMAT: {"tool": "<tool_name>"} FINAL ANSWER FORMAT: Root Cause: <Explanation of the verified issue, or 'Unknown - Diagnostic Failed' if an error occurred> Evidence: - <DiagnosticToolName> diagnostic returned: "<Diagnostic output>" Remediation: - Function: <RemediationToolName, or 'None' if in cool down> - Type: <Remediation Category, e.g., Human Education, Configuration, Patch, or 'N/A'> - Execution Result: <Remediation output, or 'Diagnostic failed, cool down initiated'> - Status: <EXECUTED, or FAILED>'}, {'role': 'user', 'content': 'shap_features:\n' + ''.join((f'- {f}: {v:.3f}\n' for f, v in zip(features, scores)))}]
#     for i, (f, tool, res, kind) in enumerate(steps, 1):
#         classification = FEATURES[f][0][0]
#         diagnostic_tools = mapped(f)
#         lines = [f'THOUGHT {i}:']
#         if i == 1:
#             lines += [f'The highest SHAP feature is {f}.', '', f'Looking up {f}', f'→ It is {classification}']
#         else:
#             pf, pt, pr, pk = steps[i - 2]
#             lines += [f'''Previous tool {pt} returned "{pr['message']}" (evaluation: {pr['status']}).''', '']
#             if f != pf:
#                 lines += [f'The next highest-ranked available SHAP feature is {f}.', '', f'Looking up {f}', f'→ It is {classification}']
#             elif kind == 'diagnostic':
#                 lines.append(f'The shared feature {f} has another unexecuted mapped diagnostic.')
#         if kind == 'diagnostic':
#             first_for_feature = not any((pf == f and pk == 'diagnostic' for pf, _, _, pk in steps[:i - 1]))
#             if first_for_feature:
#                 if classification == 'SHARED':
#                     lines.append('→ It maps to the diagnostic tools: ' + ','.join(diagnostic_tools))
#                 else:
#                     lines.append('→ It maps to the diagnostic tool: ' + diagnostic_tools[0])
#             if classification == 'SHARED':
#                 ordinal = 'first' if first_for_feature else 'next'
#                 lines += ['', f'I will call the {ordinal} diagnostic tool: {tool}.']
#             else:
#                 lines += ['', f'I will call the diagnostic tool: {tool}.']
#         else:
#             lines += [f'It maps to the remediation tool: {tool}', '', f'I will call the remediation tool: {tool}.']
#         lines += ['', f'ACTION {i}:', json.dumps({'tool': tool}, separators=(',', ':'))]
#         messages += [{'role': 'assistant', 'content': '\n'.join(lines)}, {'role': 'user', 'content': '[TOOL_RESPONSE]\n' + json.dumps({'result': res['message']}, ensure_ascii=False)}]
#     last_f, last_tool, last_result, last_kind = steps[-1]
#     if last_tool == 'UpdateDrivers' and last_result['status'] == 'POSITIVE':
#         final_status = 'EXECUTED'
#         explanation = 'UpdateDrivers performs diagnosis and remediation in one call. No additional remediation action is required.'
#     elif last_kind == 'remediation' and last_result['status'] == 'SUCCESS':
#         final_status = 'EXECUTED'
#         explanation = 'The mapped remediation completed successfully.'
#     elif last_kind == 'remediation':
#         final_status = 'REMEDIATION_FAILED'
#         explanation = 'The remediation did not complete successfully.'
#     else:
#         final_status = 'NOT_EXECUTED'
#         explanation = 'No diagnostic confirmed a condition requiring a completed remediation.'
#     detail = '\n'.join((f"- {t}: {r['status']}; {r['message']}; feature: {f}" for f, t, r, _ in steps))
#     conclusion = f'''THOUGHT {len(steps) + 1}:\nPrevious tool {last_tool} returned "{last_result['message']}" (evaluation: {last_result['status']}).\n\n{explanation}\nEvaluate observed evidence and stop execution.\n\nFINAL ANSWER:\n\nDiagnostic Results:\n{detail}\n\nRoot Cause:\n''' + ('A diagnostic condition was confirmed or driver remediation completed.' if final_status == 'EXECUTED' else 'No successful remediation was confirmed.') + f'\n\nRemediation:\n{explanation}\n\nStatus: {final_status}'
#     messages.append({'role': 'assistant', 'content': conclusion})
#     manifest = {'pattern': pattern, 'actions': [{'feature': f, 'tool': t, 'kind': k, 'status': r['status'], 'result': r['message']} for f, t, r, k in steps], 'final_status': final_status}
#     return ({'messages': messages}, manifest)

In [9]:

# def build(pattern, steps, rng):
#     """steps: list of (feature, tool, response-dict, kind)."""

#     used = set()

#     for f, t, result, kind in steps:
#         assert f in FEATURES
#         assert t not in used, f'Repeated tool: {t}'
#         used.add(t)

#         if kind == 'diagnostic':
#             assert t in mapped(f)
#             assert result in DIAG[t]
#         else:
#             assert t in REM and result in REM[t]
#             assert any(
#                 k == 'remediation' and v == t
#                 for _, k, v in FEATURES[f]
#             )

#     features = []

#     for f, _, _, _ in steps:
#         if f not in features:
#             features.append(f)

#     scores = [
#         round(0.95 - i * 0.11, 3)
#         for i in range(len(features))
#     ]

#     # ============================================================
#     # SYSTEM PROMPT AND SHAP FEATURES
#     # ============================================================

#     system_prompt = """You are the Reasoning & Execution engine for Battery Quick Discharging issues.

# Your role is to analyze SHAP feature attributions, execute diagnostics & remediations when issues are confirmed, and report the final outcome.

# CORE RULES - REACT SEQUENCE:
# Number each step sequentially (THOUGHT N:, ACTION N:).
# Do not emit a subsequent THOUGHT until you receive the output of your ACTION.

# TOOL CADENCE:
# Call only one tool per turn.
# Never repeat a diagnostic tool that has already run.

# TOOL OBSERVATION:
# Observe the diagnostic tool output thoroughly and classify it as CONFIRMED or NOT CONFIRMED.
# If NOT CONFIRMED, fall through to the next SHAP-ranked feature.

# AUTO-REMEDIATION:
# Once a diagnostic tool confirms a symptom, proceed immediately to invoke the corresponding remediation tool.

# REASONING PROCESS:

# 1. Hypothesis & Diagnostics:
# Select the appropriate read-only diagnostic tool to verify the root cause, and execute it.

# 2. Evidence Evaluation & Action:
# Read the diagnostic observation.
# If an issue is verified, explain the underlying root cause and execute the corresponding corrective remediation tool (e.g., configuration fix, update, or user notification).

# If a diagnostic tool returns an error, fails to execute, or states the issue is unconfirmed, do not invoke a remediation tool.

# Enter a cool down state (do not retry the failed tool), log the failure, and proceed with the next feature based on SHAP ranking.

# 3. Resolution:
# Confirm the remediation outcome (or report the diagnostic failure) and output your final assessment matching the layout below.

# TOOL FORMAT:
# {"tool": "<tool_name>"}

# FINAL ANSWER FORMAT:

# Root Cause: <Explanation of the verified issue, or 'Unknown - Diagnostic Failed' if an error occurred>

# Evidence:
# - <DiagnosticToolName> diagnostic returned: "<Diagnostic output>"

# Remediation:
# - Function: <RemediationToolName, or 'None' if in cool down>
# - Type: <Remediation Category, e.g., Human Education, Configuration, Patch, or 'N/A'>
# - Execution Result: <Remediation output, or 'Diagnostic failed, cool down initiated'>
# - Status: <EXECUTED, or FAILED>
# """

#     shap_content = 'shap_features:\n' + ''.join(
#         f'- {f}: {v:.3f}\n'
#         for f, v in zip(features, scores)
#     )

#     messages = [
#         {
#             'role': 'system',
#             'content': system_prompt
#         },
#         {
#             'role': 'user',
#             'content': shap_content
#         }
#     ]

#     # ============================================================
#     # GENERATE REACT STEPS
#     # ============================================================

#     for i, (f, tool, res, kind) in enumerate(steps, 1):

#         classification = FEATURES[f][0][0]
#         diagnostic_tools = mapped(f)

#         lines = [f'THOUGHT {i}:']

#         if i == 1:
#             lines += [
#                 f'The highest SHAP feature is {f}.',
#                 '',
#                 f'Looking up {f}',
#                 f'→ It is {classification}'
#             ]

#         else:
#             pf, pt, pr, pk = steps[i - 2]

#             lines += [
#                 f'Previous tool {pt} returned "{pr["message"]}" '
#                 f'(evaluation: {pr["status"]}).',
#                 ''
#             ]

#             if f != pf:
#                 lines += [
#                     f'The next highest-ranked available SHAP feature is {f}.',
#                     '',
#                     f'Looking up {f}',
#                     f'→ It is {classification}'
#                 ]

#             elif kind == 'diagnostic':
#                 lines.append(
#                     f'The shared feature {f} has another unexecuted mapped diagnostic.'
#                 )

#         if kind == 'diagnostic':

#             first_for_feature = not any(
#                 pf == f and pk == 'diagnostic'
#                 for pf, _, _, pk in steps[:i - 1]
#             )

#             if first_for_feature:

#                 if classification == 'SHARED':
#                     lines.append(
#                         '→ It maps to the diagnostic tools: '
#                         + ','.join(diagnostic_tools)
#                     )
#                 else:
#                     lines.append(
#                         '→ It maps to the diagnostic tool: '
#                         + diagnostic_tools[0]
#                     )

#             if classification == 'SHARED':
#                 ordinal = 'first' if first_for_feature else 'next'

#                 lines += [
#                     '',
#                     f'I will call the {ordinal} diagnostic tool: {tool}.'
#                 ]

#             else:
#                 lines += [
#                     '',
#                     f'I will call the diagnostic tool: {tool}.'
#                 ]

#         else:
#             lines += [
#                 f'It maps to the remediation tool: {tool}',
#                 '',
#                 f'I will call the remediation tool: {tool}.'
#             ]

#         lines += [
#             '',
#             f'ACTION {i}:',
#             json.dumps(
#                 {'tool': tool},
#                 separators=(',', ':')
#             )
#         ]

#         messages += [
#             {
#                 'role': 'assistant',
#                 'content': '\n'.join(lines)
#             },
#             {
#                 'role': 'user',
#                 'content': '[TOOL_RESPONSE]\n' + json.dumps(
#                     {'result': res['message']},
#                     ensure_ascii=False
#                 )
#             }
#         ]

#     # ============================================================
#     # FINAL RESULT EVALUATION
#     # ============================================================

#     last_f, last_tool, last_result, last_kind = steps[-1]

#     if (
#         last_tool == 'UpdateDrivers'
#         and last_result['status'] == 'POSITIVE'
#     ):
#         final_status = 'EXECUTED'
#         explanation = (
#             'UpdateDrivers performs diagnosis and remediation in one call. '
#             'No additional remediation action is required.'
#         )

#     elif (
#         last_kind == 'remediation'
#         and last_result['status'] == 'SUCCESS'
#     ):
#         final_status = 'EXECUTED'
#         explanation = 'The mapped remediation completed successfully.'

#     elif last_kind == 'remediation':
#         final_status = 'REMEDIATION_FAILED'
#         explanation = 'The remediation did not complete successfully.'

#     else:
#         final_status = 'NOT_EXECUTED'
#         explanation = (
#             'No diagnostic confirmed a condition requiring '
#             'a completed remediation.'
#         )

#     # ============================================================
#     # DIAGNOSTIC DETAILS
#     # ============================================================

#     detail = '\n'.join(
#         f"- {t}: {r['status']}; {r['message']}; feature: {f}"
#         for f, t, r, _ in steps
#     )

#     root_cause = (
#         'A diagnostic condition was confirmed or driver remediation completed.'
#         if final_status == 'EXECUTED'
#         else 'No successful remediation was confirmed.'
#     )

#     conclusion = (
#         f'THOUGHT {len(steps) + 1}:\n'
#         f'Previous tool {last_tool} returned '
#         f'"{last_result["message"]}" '
#         f'(evaluation: {last_result["status"]}).\n\n'
#         f'{explanation}\n'
#         'Evaluate observed evidence and stop execution.\n\n'
#         'FINAL ANSWER:\n\n'
#         'Diagnostic Results:\n'
#         f'{detail}\n\n'
#         'Root Cause:\n'
#         f'{root_cause}\n\n'
#         'Remediation:\n'
#         f'{explanation}\n\n'
#         f'Status: {final_status}'
#     )

#     messages.append({
#         'role': 'assistant',
#         'content': conclusion
#     })

#     # ============================================================
#     # MANIFEST
#     # ============================================================

#     manifest = {
#         'pattern': pattern,
#         'actions': [
#             {
#                 'feature': f,
#                 'tool': t,
#                 'kind': k,
#                 'status': r['status'],
#                 'result': r['message']
#             }
#             for f, t, r, k in steps
#         ],
#         'final_status': final_status
#     }

#     return (
#         {'messages': messages},
#         manifest
#     )


In [10]:

import json

def build(pattern, steps, rng):
    """steps: list of (feature, tool, response-dict, kind)."""

    if not steps:
        raise ValueError("steps must contain at least one action.")

    used = set()

    for f, t, result, kind in steps:
        assert f in FEATURES
        assert t not in used, f'Repeated tool: {t}'
        used.add(t)

        if kind == 'diagnostic':
            assert t in mapped(f)
            assert result in DIAG[t]
        else:
            assert t in REM and result in REM[t]
            assert any(
                k == 'remediation' and v == t
                for _, k, v in FEATURES[f]
            )

    # ============================================================
    # SHAP FEATURE GENERATION - MINIMUM 3 FEATURES
    # ============================================================

    MIN_SHAP_FEATURES = 3

    features = []

    # Preserve the order of features used in steps
    for f, _, _, _ in steps:
        if f not in features:
            features.append(f)

    # Add other available features if fewer than 3 exist
    if len(features) < MIN_SHAP_FEATURES:

        available_features = [
            f for f in FEATURES
            if f not in features
        ]

        required = MIN_SHAP_FEATURES - len(features)

        if len(available_features) < required:
            raise ValueError(
                f"Cannot generate {MIN_SHAP_FEATURES} SHAP features. "
                "Not enough unique features in FEATURES."
            )

        additional_features = rng.sample(
            available_features,
            required
        )

        features.extend(additional_features)

    # Generate descending SHAP scores
    scores = [
        round(0.95 - i * 0.11, 3)
        for i in range(len(features))
    ]

    shap_content = 'shap_features:\n' + ''.join(
        f'- {f}: {v:.3f}\n'
        for f, v in zip(features, scores)
    )

    # ============================================================
    # SYSTEM PROMPT
    # ============================================================

    system_prompt = """You are the Reasoning & Execution engine for Battery Quick Discharging issues.

Your role is to analyze SHAP feature attributions, execute diagnostics & remediations when issues are confirmed, and report the final outcome.

CORE RULES - REACT SEQUENCE:
Number each step sequentially (THOUGHT N:, ACTION N:).
Do not emit a subsequent THOUGHT until you receive the output of your ACTION.

TOOL CADENCE:
Call only one tool per turn.
Never repeat a diagnostic tool that has already run.

TOOL OBSERVATION:
Observe the diagnostic tool output thoroughly and classify it as CONFIRMED or NOT CONFIRMED.
If NOT CONFIRMED, fall through to the next SHAP-ranked feature.

AUTO-REMEDIATION:
Once a diagnostic tool confirms a symptom, proceed immediately to invoke the corresponding remediation tool.

REASONING PROCESS:

1. Hypothesis & Diagnostics:
Select the appropriate read-only diagnostic tool to verify the root cause, and execute it.

2. Evidence Evaluation & Action:
Read the diagnostic observation.
If an issue is verified, explain the underlying root cause and execute the corresponding corrective remediation tool (e.g., configuration fix, update, or user notification).

If a diagnostic tool returns an error, fails to execute, or states the issue is unconfirmed, do not invoke a remediation tool.

Enter a cool down state (do not retry the failed tool), log the failure, and proceed with the next feature based on SHAP ranking.

3. Resolution:
Confirm the remediation outcome (or report the diagnostic failure) and output your final assessment matching the layout below.

TOOL FORMAT:
{"tool": "<tool_name>"}

FINAL ANSWER FORMAT:

Root Cause: <Explanation of the verified issue, or 'Unknown - Diagnostic Failed' if an error occurred>

Evidence:
- <DiagnosticToolName> diagnostic returned: "<Diagnostic output>"

Remediation:
- Function: <RemediationToolName, or 'None' if in cool down>
- Type: <Remediation Category, e.g., Human Education, Configuration, Patch, or 'N/A'>
- Execution Result: <Remediation output, or 'Diagnostic failed, cool down initiated'>
- Status: <EXECUTED, or FAILED>
"""

    messages = [
        {
            'role': 'system',
            'content': system_prompt
        },
        {
            'role': 'user',
            'content': shap_content
        }
    ]

    # ============================================================
    # GENERATE REACT STEPS
    # ============================================================

    for i, (f, tool, res, kind) in enumerate(steps, 1):

        classification = FEATURES[f][0][0]
        diagnostic_tools = mapped(f)

        lines = [f'THOUGHT {i}:']

        if i == 1:
            lines += [
                f'The highest SHAP feature is {f}.',
                '',
                f'Looking up {f}',
                f'→ It is {classification}'
            ]

        else:
            pf, pt, pr, pk = steps[i - 2]

            lines += [
                f'Previous tool {pt} returned "{pr["message"]}" '
                f'(evaluation: {pr["status"]}).',
                ''
            ]

            if f != pf:
                lines += [
                    f'The next highest-ranked available SHAP feature is {f}.',
                    '',
                    f'Looking up {f}',
                    f'→ It is {classification}'
                ]

            elif kind == 'diagnostic':
                lines.append(
                    f'The shared feature {f} has another unexecuted mapped diagnostic.'
                )

        if kind == 'diagnostic':

            first_for_feature = not any(
                pf == f and pk == 'diagnostic'
                for pf, _, _, pk in steps[:i - 1]
            )

            if first_for_feature:

                if classification == 'SHARED':
                    lines.append(
                        '→ It maps to the diagnostic tools: '
                        + ','.join(diagnostic_tools)
                    )
                else:
                    lines.append(
                        '→ It maps to the diagnostic tool: '
                        + diagnostic_tools[0]
                    )

            if classification == 'SHARED':
                ordinal = 'first' if first_for_feature else 'next'

                lines += [
                    '',
                    f'I will call the {ordinal} diagnostic tool: {tool}.'
                ]

            else:
                lines += [
                    '',
                    f'I will call the diagnostic tool: {tool}.'
                ]

        else:
            lines += [
                f'It maps to the remediation tool: {tool}',
                '',
                f'I will call the remediation tool: {tool}.'
            ]

        lines += [
            '',
            f'ACTION {i}:',
            json.dumps(
                {'tool': tool},
                separators=(',', ':')
            )
        ]

        messages += [
            {
                'role': 'assistant',
                'content': '\n'.join(lines)
            },
            {
                'role': 'user',
                'content': '[TOOL_RESPONSE]\n' + json.dumps(
                    {'result': res['message']},
                    ensure_ascii=False
                )
            }
        ]

    # ============================================================
    # FINAL RESULT EVALUATION
    # ============================================================

    last_f, last_tool, last_result, last_kind = steps[-1]

    if (
        last_tool == 'UpdateDrivers'
        and last_result['status'] == 'POSITIVE'
    ):
        final_status = 'EXECUTED'
        explanation = (
            'UpdateDrivers performs diagnosis and remediation in one call. '
            'No additional remediation action is required.'
        )

    elif (
        last_kind == 'remediation'
        and last_result['status'] == 'SUCCESS'
    ):
        final_status = 'EXECUTED'
        explanation = 'The mapped remediation completed successfully.'

    elif last_kind == 'remediation':
        final_status = 'REMEDIATION_FAILED'
        explanation = 'The remediation did not complete successfully.'

    else:
        final_status = 'NOT_EXECUTED'
        explanation = (
            'No diagnostic confirmed a condition requiring '
            'a completed remediation.'
        )

    # ============================================================
    # DIAGNOSTIC DETAILS
    # ============================================================

    detail = '\n'.join(
        f"- {t}: {r['status']}; {r['message']}; feature: {f}"
        for f, t, r, _ in steps
    )

    root_cause = (
        'A diagnostic condition was confirmed or driver remediation completed.'
        if final_status == 'EXECUTED'
        else 'No successful remediation was confirmed.'
    )

    conclusion = (
        f'THOUGHT {len(steps) + 1}:\n'
        f'Previous tool {last_tool} returned '
        f'"{last_result["message"]}" '
        f'(evaluation: {last_result["status"]}).\n\n'
        f'{explanation}\n'
        'Evaluate observed evidence and stop execution.\n\n'
        'FINAL ANSWER:\n\n'
        'Diagnostic Results:\n'
        f'{detail}\n\n'
        'Root Cause:\n'
        f'{root_cause}\n\n'
        'Remediation:\n'
        f'{explanation}\n\n'
        f'Status: {final_status}'
    )

    messages.append({
        'role': 'assistant',
        'content': conclusion
    })

    # ============================================================
    # MANIFEST
    # ============================================================

    manifest = {
        'pattern': pattern,
        'actions': [
            {
                'feature': f,
                'tool': t,
                'kind': k,
                'status': r['status'],
                'result': r['message']
            }
            for f, t, r, k in steps
        ],
        'final_status': final_status
    }

    return (
        {'messages': messages},
        manifest
    )


### 5c. Coverage patterns

In [11]:
def construct_patterns(rng):
    """Small deterministic patterns; shared candidates follow full mapping order."""
    patterns = []

    def trace(name, feature_tools, statuses, remediation_status='SUCCESS'):
        assert len(feature_tools) == len(statuses)
        steps = []
        for (f, tool), status in zip(feature_tools, statuses):
            steps.append((f, tool, response(tool, status, rng), 'diagnostic'))
            if status == 'POSITIVE' and tool != 'UpdateDrivers':
                rtool = PAIR[tool]
                steps.append((f, rtool, remediation_response(rtool, remediation_status, rng), 'remediation'))
                break
            if tool == 'UpdateDrivers' and status == 'POSITIVE':
                break
        patterns.append((name, steps))

    def exclusive(name, tools, statuses):
        trace(name, [(feature_for(t, 'EXCLUSIVE'), t) for t in tools], statuses)
    exclusive('base_positive', ['CheckLowEnergySettings'], ['POSITIVE'])
    exclusive('fallthrough_second', ['CheckLowEnergySettings', 'CheckBatteryChargeCap'], ['NEGATIVE', 'POSITIVE'])
    exclusive('diagnostic_error_fallthrough', ['CheckLowEnergySettings', 'CheckBatteryChargeCap'], ['ERROR', 'POSITIVE'])
    exclusive('fallthrough_third', ['CheckLowEnergySettings', 'CheckBatteryChargeCap', 'CheckHeavyBatteryUsageByApps'], ['NEGATIVE', 'ERROR', 'POSITIVE'])
    exclusive('all_features_exhausted', ['CheckLowEnergySettings', 'CheckBatteryChargeCap'], ['NEGATIVE', 'ERROR'])
    exclusive('driver_update_success', ['UpdateDrivers'], ['POSITIVE'])
    exclusive('driver_diagnostic_error_then_update', ['CheckHeavyBatteryUsageByApps', 'UpdateDrivers'], ['ERROR', 'POSITIVE'])
    exclusive('driver_negative_fallthrough', ['UpdateDrivers', 'CheckBatteryChargeCap'], ['NEGATIVE', 'POSITIVE'])
    exclusive('driver_error_fallthrough', ['UpdateDrivers', 'CheckBatteryChargeCap'], ['ERROR', 'POSITIVE'])
    bios = 'BIOS_pass'
    bt = mapped(bios)
    trace('shared_error_then_second_mapped_positive', [(bios, t) for t in bt], ['ERROR', 'POSITIVE'])
    trace('shared_negative_then_second_mapped_positive', [(bios, t) for t in bt], ['NEGATIVE', 'POSITIVE'])
    trace('shared_all_mapped_diagnostics_exhausted', [(bios, t) for t in bt], ['NEGATIVE', 'ERROR'])
    charge = 'CurrentChargePercentage'
    ct = mapped(charge)
    driver_idx = ct.index('UpdateDrivers')
    prefix = [(charge, t) for t in ct[:driver_idx + 1]]
    trace('driver_shared_after_error', prefix, ['ERROR'] + ['NEGATIVE'] * (len(prefix) - 2) + ['POSITIVE'])
    trace('driver_shared_after_negative', prefix, ['NEGATIVE'] * (len(prefix) - 1) + ['POSITIVE'])
    trace('shared_full_diagnostic_fallthrough', [(charge, t) for t in ct], ['NEGATIVE'] * len(ct))
    f0 = feature_for('CheckLowEnergySettings', 'EXCLUSIVE')
    for name, first in [('shared_exclusive_negative_then_shared_positive', 'NEGATIVE'), ('shared_exclusive_error_then_shared_sequential', 'ERROR')]:
        trace(name, [(f0, 'CheckLowEnergySettings')] + [(bios, t) for t in bt], [first, 'ERROR', 'POSITIVE'])
    f1 = feature_for('CheckLowEnergySettings', 'EXCLUSIVE')
    f2 = feature_for('CheckBatteryChargeCap', 'EXCLUSIVE')
    for name, outcome in [('approach2_remediation_negative', 'NEGATIVE'), ('approach2_remediation_error', 'ERROR')]:
        patterns.append((name, [(f1, 'CheckLowEnergySettings', response('CheckLowEnergySettings', 'POSITIVE', rng), 'diagnostic'), (f1, 'SetLowEnergySettings', remediation_response('SetLowEnergySettings', outcome, rng), 'remediation'), (f2, 'CheckBatteryChargeCap', response('CheckBatteryChargeCap', 'POSITIVE', rng), 'diagnostic'), (f2, 'SetBatteryChargeCap', remediation_response('SetBatteryChargeCap', 'SUCCESS', rng), 'remediation')]))
    # Terminal remediation failure remains allowed after the second SHAP
    # feature has been evaluated. The first diagnostic is unconfirmed;
    # the second is confirmed but its remediation fails.
    for rem_status in ('NEGATIVE', 'ERROR'):
        first_f = feature_for('CheckLowEnergySettings', 'EXCLUSIVE')
        second_f = feature_for('CheckHeavyBatteryUsageByApps', 'EXCLUSIVE')
        steps = [
            (first_f, 'CheckLowEnergySettings', response('CheckLowEnergySettings', 'NEGATIVE', rng), 'diagnostic'),
            (second_f, 'CheckHeavyBatteryUsageByApps', response('CheckHeavyBatteryUsageByApps', 'POSITIVE', rng), 'diagnostic'),
            (second_f, 'NotifyHeavyBatteryUsageByApps', remediation_response('NotifyHeavyBatteryUsageByApps', rem_status, rng), 'remediation'),
        ]
        patterns.append((f'second_feature_remediation_{rem_status.lower()}_terminal', steps))
    return patterns

def coverage_cases(rng):
    """One targeted case per real Sheet3/Sheet1 output, plus behavior patterns."""
    cases = []
    for tool, variants in DIAG.items():
        for variant in variants:
            f = feature_for(tool, 'EXCLUSIVE')
            steps = [(f, tool, variant, 'diagnostic')]
            if variant['status'] == 'POSITIVE' and tool != 'UpdateDrivers':
                rtool = PAIR[tool]
                steps.append((f, rtool, remediation_response(rtool, 'SUCCESS', rng), 'remediation'))
            cases.append((f"output_{tool}_{variant['status']}", steps))
    for tool, variants in REM.items():
        for variant in variants:
            f = feature_for(next((d for d, r in PAIR.items() if r == tool)), 'EXCLUSIVE')
            d = next((d for d, r in PAIR.items() if r == tool))
            steps = [(f, d, response(d, 'POSITIVE', rng), 'diagnostic'), (f, tool, variant, 'remediation')]
            cases.append((f"remediation_output_{tool}_{variant['status']}", steps))
    return cases

### 5d. Dataset writer — mandatory diagnostic fallthrough; remediation failure falls through while a second SHAP feature is available. Terminal `REMEDIATION_FAILED` is permitted after two features have been evaluated.


In [12]:
# Policy: an unconfirmed diagnostic MUST fall through; a failed remediation
# MUST fall through while an additional feature remains in the two-feature
# diagnostic budget. REMEDIATION_FAILED is permitted only once that budget
# has been used. Do not fabricate a successful result for a failed remediation.
MAX_EVALUATED_SHAP_FEATURES = 2
UNCONFIRMED = {'NEGATIVE', 'ERROR', 'COOLDOWN'}
REMEDIATION_FAILURES = {'NEGATIVE', 'ERROR', 'COOLDOWN'}

def successful_fallthrough(steps, rng):
    used_features = list(dict.fromkeys(f for f, _, _, _ in steps))
    used_tools = {t for _, t, _, _ in steps}
    candidates = []
    for feature, entries in FEATURES.items():
        if feature in used_features or entries[0][0] != 'EXCLUSIVE':
            continue
        for tool in mapped(feature):
            rem = PAIR.get(tool)
            if (tool != 'UpdateDrivers' and rem and tool not in used_tools
                and rem not in used_tools
                and any(x['status'] == 'POSITIVE' for x in DIAG[tool])
                and any(x['status'] == 'SUCCESS' for x in REM[rem])
                and any(k == 'remediation' and t == rem for _, k, t in entries)):
                candidates.append((feature, tool, rem))
    if not candidates:
        raise ValueError('No unused mapped positive diagnostic/remediation pair')
    feature, diag, rem = rng.choice(candidates)
    return steps + [
        (feature, diag, response(diag, 'POSITIVE', rng), 'diagnostic'),
        (feature, rem, remediation_response(rem, 'SUCCESS', rng), 'remediation'),
    ]

def extend_unconfirmed_trajectory(steps, rng):
    if not steps:
        raise ValueError('Empty trajectory')
    feature, tool, result, kind = steps[-1]
    used_features = list(dict.fromkeys(f for f, _, _, _ in steps))

    if kind == 'diagnostic' and result['status'] in UNCONFIRMED:
        # No terminal NOT_EXECUTED: append a mapped confirmed diagnostic and
        # successful remediation, even for legacy coverage/exhaustion patterns.
        return successful_fallthrough(steps, rng)

    if kind == 'remediation' and result['status'] in REMEDIATION_FAILURES:
        # A remediation failure must fall through if a second SHAP feature
        # has not been evaluated yet. If two or more have already been used,
        # terminal REMEDIATION_FAILED is a valid outcome.
        if len(used_features) < MAX_EVALUATED_SHAP_FEATURES:
            return successful_fallthrough(steps, rng)
        return steps

    return steps

def generate(count, seed, dataset, manifest_path, report_path):
    rng = random.Random(seed)
    required = construct_patterns(rng) + coverage_cases(rng)
    minimum = len(required)
    if count == 0:
        count = minimum
    if count < minimum:
        raise ValueError(f'count={count} cannot cover all reference outputs; minimum={minimum}')
    cases = list(required)
    while len(cases) < count:
        name, steps = rng.choice(required)
        remap = {}
        for old_f in dict.fromkeys(f for f, _, _, _ in steps):
            associated = [(t, k) for f, t, _, k in steps if f == old_f]
            options = [f for f, entries in FEATURES.items()
                       if FEATURES[f][0][0] == FEATURES[old_f][0][0]
                       and all(any(ek == k and et == t for _, ek, et in entries)
                               for t, k in associated)
                       and f not in remap.values()
                       and (FEATURES[f][0][0] != 'SHARED'
                            or [t for t in mapped(f) if t in [a for a, k in associated if k == 'diagnostic']]
                               == [a for a, k in associated if k == 'diagnostic'])]
            remap[old_f] = rng.choice(options) if options else old_f
        cases.append((name, [(remap[f], t, r, k) for f, t, r, k in steps]))
    rng.shuffle(cases)
    data, metadata = [], []
    for i, (name, steps) in enumerate(cases, 1):
        steps = extend_unconfirmed_trajectory(steps, rng)
        example, record = build(name, steps, rng)
        record['id'] = i
        data.append(example)
        metadata.append(record)
    for path, records in ((dataset, data), (manifest_path, metadata)):
        with open(path, 'w', encoding='utf-8') as fp:
            for record in records:
                fp.write(json.dumps(record, ensure_ascii=False) + '\n')
    observed = {(a['tool'], a['result']) for m in metadata for a in m['actions']}
    expected = {(t, r['message']) for catalog in (DIAG, REM)
                for t, rows in catalog.items() for r in rows}
    assert expected <= observed, 'Reference output coverage incomplete'
    statuses = Counter(m['final_status'] for m in metadata)
    assert statuses.get('NOT_EXECUTED', 0) == 0, statuses
    report = {'examples': len(data), 'minimum_full_coverage_examples': minimum,
              'diagnostic_tools': len(DIAG), 'remediation_tools': len(REM),
              'distinct_reference_outputs': len(expected),
              'covered_reference_outputs': len(expected & observed),
              'final_status_counts': dict(statuses),
              'patterns': dict(Counter(m['pattern'] for m in metadata))}
    Path(report_path).write_text(json.dumps(report, indent=2), encoding='utf-8')
    print(json.dumps({k:v for k,v in report.items() if k != 'patterns'}, indent=2))
    print(f'Dataset: {dataset}\nManifest: {manifest_path}\nCoverage: {report_path}')


## 6. Generate 3,000 examples

The generator raises if workbook-response coverage is incomplete. `pattern` stays in the **manifest**, not the training JSONL.

In [13]:
generate(COUNT, SEED, DATASET_PATH, MANIFEST_PATH, REPORT_PATH)

{
  "examples": 3000,
  "minimum_full_coverage_examples": 108,
  "diagnostic_tools": 7,
  "remediation_tools": 6,
  "distinct_workbook_tool_outputs": 89,
  "covered_workbook_tool_outputs": 89,
  "unsupported_legacy_patterns": [
    "remediation_cooldown_blocked",
    "approach1_diagnostic_cooldown",
    "approach2_remediation_cooldown",
    "all_features_exhausted_with_cooldown"
  ],
  "note": "Sheet3 diagnostic labels are authoritative; Sheet1 remediation outcomes are inferred from message text. No cooldown outputs were present in the workbook."
}
Dataset: /home/dell/SLM_Engineering/Data_Generation/Final_Data_Gen/scenario2_reference_3000_training.jsonl
Manifest: /home/dell/SLM_Engineering/Data_Generation/Final_Data_Gen/scenario2_reference_3000_manifest.jsonl
Coverage: /home/dell/SLM_Engineering/Data_Generation/Final_Data_Gen/scenario2_reference_3000_validation.json


## 7. Independent dataset validator

Checks count, message structure, one tool per assistant turn, tool-response pairing, unique diagnostics, mapped features, exact catalog strings, immediate remediation on positive diagnosis, and output coverage.

In [14]:
def validate_dataset(dataset_path, manifest_path, expected_count=3000):
    data = [json.loads(line) for line in Path(dataset_path).read_text(encoding="utf-8").splitlines() if line.strip()]
    manifests = [json.loads(line) for line in Path(manifest_path).read_text(encoding="utf-8").splitlines() if line.strip()]
    errors = []
    if len(data) != expected_count or len(manifests) != expected_count:
        errors.append(f"Count mismatch: dataset={len(data)}, manifest={len(manifests)}, expected={expected_count}")
    observed = set()
    diag_status = Counter()
    rem_status = Counter()
    tool_counts = Counter()
    patterns = Counter()
    for index, (example, manifest) in enumerate(zip(data, manifests), 1):
        def fail(msg): errors.append(f"Example {index}: {msg}")
        if set(example) != {"messages"}: fail("Training object contains unexpected keys")
        msgs = example.get("messages", [])
        actions = manifest.get("actions", [])
        if len(msgs) != 2 + 2 * len(actions) + 1: fail("Incorrect message count")
        if not msgs or msgs[0].get("role") != "system" or msgs[1].get("role") != "user":
            fail("Missing initial system/user messages")
        if not msgs or msgs[-1].get("role") != "assistant" or "FINAL ANSWER:" not in msgs[-1].get("content", ""):
            fail("Missing final assistant answer")
        used_diag = set()
        # Reject premature termination after an unconfirmed diagnostic.
        if actions:
            last = actions[-1]
            if last['kind'] == 'diagnostic' and last['status'] in ('NEGATIVE', 'ERROR', 'COOLDOWN'):
                fail('Unconfirmed diagnostic terminated trajectory instead of falling through')
            first = actions[0]
            if first['kind'] == 'diagnostic' and first['status'] in ('NEGATIVE', 'ERROR', 'COOLDOWN'):
                if len(actions) < 2:
                    fail('First diagnostic unconfirmed: no next tool action')
                elif not any(a['kind'] == 'diagnostic' for a in actions[1:]):
                    fail('First diagnostic unconfirmed: no subsequent diagnostic')
                # A SHARED feature may legitimately execute another mapped
                # diagnostic before moving to the next ranked SHAP feature.
        if actions:
            last = actions[-1]
            evaluated_features = list(dict.fromkeys(a['feature'] for a in actions))
            if manifest.get('final_status') == 'NOT_EXECUTED':
                fail('Terminal NOT_EXECUTED forbidden')
            if last['kind'] == 'remediation' and last['status'] in REMEDIATION_FAILURES:
                if len(evaluated_features) < MAX_EVALUATED_SHAP_FEATURES:
                    fail('Remediation failure terminated without second-feature fallthrough')
                if manifest.get('final_status') != 'REMEDIATION_FAILED':
                    fail('Terminal remediation failure must be REMEDIATION_FAILED')
            elif last['kind'] == 'remediation' and last['status'] == 'SUCCESS':
                if manifest.get('final_status') != 'EXECUTED':
                    fail('Successful remediation must be EXECUTED')
            elif last['tool'] == 'UpdateDrivers' and last['status'] == 'POSITIVE':
                if manifest.get('final_status') != 'EXECUTED':
                    fail('Successful driver update must be EXECUTED')
            for j, a in enumerate(actions[:-1]):
                if a['kind'] == 'remediation' and a['status'] in REMEDIATION_FAILURES:
                    following = actions[j+1:]
                    if not following or following[0]['kind'] != 'diagnostic':
                        fail('Failed remediation did not advance to another diagnostic')
                    elif following[0]['feature'] == a['feature']:
                        fail('Failed remediation did not advance to next SHAP feature')
        patterns[manifest.get("pattern", "UNKNOWN")] += 1
        for j, action in enumerate(actions):
            feature, tool, kind, status, result = (action[k] for k in ("feature", "tool", "kind", "status", "result"))
            observed.add((tool, result))
            tool_counts[tool] += 1
            if feature not in FEATURES: fail(f"Unknown feature {feature}"); continue
            mapped_entries = FEATURES[feature]
            if not any(k == kind and t == tool for _, k, t in mapped_entries):
                fail(f"Feature/tool mismatch: {feature}/{kind}/{tool}")
            if kind == "diagnostic":
                diag_status[(tool, status)] += 1
                if tool in used_diag: fail(f"Repeated diagnostic {tool}")
                used_diag.add(tool)
                catalog = DIAG.get(tool, [])
                if status == "POSITIVE" and tool != "UpdateDrivers":
                    if j+1 >= len(actions) or actions[j+1]["kind"] != "remediation" or actions[j+1]["tool"] != PAIR[tool]:
                        fail(f"Positive diagnostic {tool} not immediately remediated")
                if tool == "UpdateDrivers" and status == "POSITIVE" and j+1 < len(actions):
                    fail("Positive UpdateDrivers must end tool execution")
            else:
                rem_status[(tool, status)] += 1
                catalog = REM.get(tool, [])
                if j == 0 or actions[j-1]["kind"] != "diagnostic" or actions[j-1]["status"] != "POSITIVE" or PAIR.get(actions[j-1]["tool"]) != tool:
                    fail(f"Remediation {tool} has no preceding positive diagnostic")
            if not any(r["status"] == status and r["message"] == result for r in catalog):
                fail(f"Non-reference response: {tool}/{status}/{result}")
            assistant_idx = 2 + 2*j
            if assistant_idx + 1 < len(msgs):
                assistant_msg, tool_msg = msgs[assistant_idx:assistant_idx+2]
                if assistant_msg.get("role") != "assistant" or tool_msg.get("role") != "user":
                    fail(f"Invalid turn roles at action {j+1}")
                if f"ACTION {j+1}:" not in assistant_msg.get("content", "") or f"THOUGHT {j+1}:" not in assistant_msg.get("content", ""):
                    fail(f"Incorrect ReAct numbering at action {j+1}")
                try:
                    tool_call = json.loads(assistant_msg["content"].split(f"ACTION {j+1}:", 1)[1].strip())
                    tool_result = json.loads(tool_msg["content"].split("[TOOL_RESPONSE]", 1)[1].strip())
                    if tool_call != {"tool": tool} or tool_result != {"result": result}:
                        fail(f"Tool call/response mismatch at action {j+1}")
                except (ValueError, IndexError, KeyError) as exc:
                    fail(f"Malformed tool exchange at action {j+1}: {exc}")
    expected = {(t, r["message"]) for catalog in (DIAG, REM) for t, rows in catalog.items() for r in rows}
    missing = sorted(expected - observed)
    if missing: errors.append(f"Missing {len(missing)} reference tool outputs: {missing[:5]}")
    report = {
        "passed": not errors,
        "examples": len(data),
        "expected_examples": expected_count,
        "reference_outputs_covered": len(expected & observed),
        "reference_outputs_total": len(expected),
        "diagnostic_status_counts": {f"{t} | {s}": n for (t,s),n in sorted(diag_status.items())},
        "remediation_status_counts": {f"{t} | {s}": n for (t,s),n in sorted(rem_status.items())},
        "tool_counts": dict(tool_counts),
        "pattern_counts": dict(patterns),
        "unsupported_legacy_patterns": UNSUPPORTED_LEGACY_PATTERNS,
        "errors": errors[:100],
        "total_errors": len(errors),
    }
    return report


## 8. Run validator and save report

In [15]:
VALIDATION = validate_dataset(DATASET_PATH, MANIFEST_PATH, COUNT)
REPORT_PATH.write_text(json.dumps(VALIDATION, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps({k: v for k, v in VALIDATION.items() if k not in ("diagnostic_status_counts", "remediation_status_counts", "pattern_counts", "tool_counts")}, indent=2))
assert VALIDATION["passed"], "Validation failed; inspect the report"

{
  "passed": true,
  "examples": 3000,
  "expected_examples": 3000,
  "reference_outputs_covered": 89,
  "reference_outputs_total": 89,
  "unsupported_legacy_patterns": [
    "remediation_cooldown_blocked",
    "approach1_diagnostic_cooldown",
    "approach2_remediation_cooldown",
    "all_features_exhausted_with_cooldown"
  ],
  "errors": [],
  "total_errors": 0
}


## 9. Outcome distribution

In [16]:
print("DIAGNOSTIC OUTCOMES")
for k, v in sorted(VALIDATION["diagnostic_status_counts"].items()):
    print(f"{k:65s} {v:5d}")
print("\nREMEDIATION OUTCOMES")
for k, v in sorted(VALIDATION["remediation_status_counts"].items()):
    print(f"{k:65s} {v:5d}")

DIAGNOSTIC OUTCOMES
CheckBatteryChargeCap | ERROR                                       171
CheckBatteryChargeCap | NEGATIVE                                    170
CheckBatteryChargeCap | POSITIVE                                    264
CheckHeavyACAdapterUsageSettings | ERROR                             63
CheckHeavyACAdapterUsageSettings | NEGATIVE                          30
CheckHeavyACAdapterUsageSettings | POSITIVE                         234
CheckHeavyBatteryUsageByApps | ERROR                                 97
CheckHeavyBatteryUsageByApps | NEGATIVE                             110
CheckHeavyBatteryUsageByApps | POSITIVE                             303
CheckLowEnergySettings | ERROR                                       75
CheckLowEnergySettings | NEGATIVE                                   188
CheckLowEnergySettings | POSITIVE                                   502
GetACAdapterChargeWattageStatus | ERROR                              58
GetACAdapterChargeWattageStatus | NEGATIVE  